# Preprocessing Encoding Data Abu Vulkanik (Train/Val/Test)
Notebook ini melakukan preprocessing berbasis split agar tidak terjadi data leakage.
1. Load data train, validation, dan test dari file terpisah
2. Fit preprocessing hanya pada train
3. Transform train/val/test ke format siap-model
4. Simpan hasil preprocessing dan artefak mapping

In [1]:
from pathlib import Path
from time import perf_counter
import json
import pandas as pd

pd.set_option('display.max_columns', None)
process_times = {}

In [2]:
# Konfigurasi input/output
step_start = perf_counter()

INPUT_DIR = Path('.')
TRAIN_PATH = INPUT_DIR / 'java_ash_train_augmented.csv'
VAL_PATH = INPUT_DIR / 'java_ash_val.csv'
TEST_PATH = INPUT_DIR / 'java_ash_test.csv'

OUTPUT_DIR = Path('preprocessing_outputs_aug_v2')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ARTIFACT_PATH = OUTPUT_DIR / 'preprocessing_artifacts.json'

for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    if not p.exists():
        raise FileNotFoundError(f'File tidak ditemukan: {p}')

process_times['setup_paths'] = perf_counter() - step_start
print(f"Waktu setup_paths: {process_times['setup_paths']:.6f} detik")
print('Input files:')
print('-', TRAIN_PATH)
print('-', VAL_PATH)
print('-', TEST_PATH)
print('Output dir:', OUTPUT_DIR)

Waktu setup_paths: 0.000984 detik
Input files:
- java_ash_train_augmented.csv
- java_ash_val.csv
- java_ash_test.csv
Output dir: preprocessing_outputs_aug_v2


In [3]:
# Load split data
step_start = perf_counter()

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

required_cols = ['timestamp', 'alert_level', 'volcano_filter']
for split_name, split_df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    missing = [c for c in required_cols if c not in split_df.columns]
    if missing:
        raise ValueError(f'Kolom wajib tidak ada di {split_name}: {missing}')

process_times['load_data'] = perf_counter() - step_start
print(f"Waktu load_data: {process_times['load_data']:.6f} detik")
print('Shape train:', train_df.shape)
print('Shape val  :', val_df.shape)
print('Shape test :', test_df.shape)
train_df.head()

Waktu load_data: 0.039766 detik
Shape train: (8000, 16)
Shape val  : (341, 16)
Shape test : (342, 16)


,id,timestamp,volcano_filter,alert_level,latitude,longitude,elevation,tinggi_letusan_m,kec_angin_km_jam,arah_angin_deg,amplitudo,duration,jarak_km,luas_km2,sudut_deg,radius_km
0,1652,2021-02-09 23:43:00 UTC,Raung,Orange,-8.123735,114.039080,3332,1278.657175,7.045481,231.395353,12.621366,121.878782,34.494420,0.000000,309.171766,9.513491
1,1667,2025-06-17 10:56:00 UTC,Raung,Orange,-8.123314,114.038575,3332,19724.534182,7.991023,129.588183,5.237928,119.207849,50.535607,185.610679,350.568295,25.311322
2,1551,2025-07-19 02:40:00 UTC,Semeru,Orange,-8.105096,112.918359,3676,596.955913,0.695739,193.864938,22.417334,27.752157,12.836212,479.763599,6.915086,4.513697
3,1693,2021-01-22 00:13:00 UTC,Raung,Yellow,-8.124275,114.040547,3332,20856.325769,12.042612,85.595558,13.915070,126.255824,27.142432,350.362405,252.424605,21.819699
4,1632,2016-04-03 04:21:00 UTC,Bromo,Orange,-7.941559,112.951928,2329,8656.536600,6.967262,136.540679,8.980861,124.002421,85.911765,817.289461,218.915146,22.362665


In [4]:
# Fit preprocessing di train, lalu transform train/val/test
step_start = perf_counter()

def preprocess_split(df, alert_mapping, volcano_categories):
    result = df.copy()

    # timestamp -> bulan (1-12)
    ts = pd.to_datetime(result['timestamp'], utc=True, errors='coerce')
    result['timestamp'] = ts.dt.month.fillna(0).astype(int)

    # alert_level -> label encoding dari mapping train
    result['alert_level'] = result['alert_level'].map(alert_mapping).fillna(-1).astype(int)

    # volcano_filter -> one hot dengan kategori train
    volcano_categorical = pd.Categorical(result['volcano_filter'], categories=volcano_categories)
    volcano_ohe = pd.get_dummies(volcano_categorical, prefix='volcano_filter', dtype=int)

    result = pd.concat([result.drop(columns=['volcano_filter']), volcano_ohe], axis=1)
    return result

alert_mapping = {
    label: idx for idx, label in enumerate(sorted(train_df['alert_level'].dropna().unique()))
}
volcano_categories = sorted(train_df['volcano_filter'].dropna().unique())

train_encoded = preprocess_split(train_df, alert_mapping, volcano_categories)
val_encoded = preprocess_split(val_df, alert_mapping, volcano_categories)
test_encoded = preprocess_split(test_df, alert_mapping, volcano_categories)

process_times['preprocess_encoding'] = perf_counter() - step_start
print(f"Waktu preprocess_encoding: {process_times['preprocess_encoding']:.6f} detik")
print('Mapping alert_level (train):', alert_mapping)
print('Kategori volcano_filter (train):', volcano_categories)
print('Shape encoded train:', train_encoded.shape)
print('Shape encoded val  :', val_encoded.shape)
print('Shape encoded test :', test_encoded.shape)

Waktu preprocess_encoding: 0.170042 detik
Mapping alert_level (train): {'Green': 0, 'Orange': 1, 'Red': 2, 'Yellow': 3}
Kategori volcano_filter (train): ['Bromo', 'Merapi', 'Raung', 'Semeru']
Shape encoded train: (8000, 19)
Shape encoded val  : (341, 19)
Shape encoded test : (342, 19)


In [5]:
# Simpan hasil preprocessing dan artefak
step_start = perf_counter()

train_output = OUTPUT_DIR / 'java_ash_train_augmented_encoded.csv'
val_output = OUTPUT_DIR / 'java_ash_val_encoded.csv'
test_output = OUTPUT_DIR / 'java_ash_test_encoded.csv'

train_encoded.to_csv(train_output, index=False)
val_encoded.to_csv(val_output, index=False)
test_encoded.to_csv(test_output, index=False)

artifacts = {
    'input_files': {
        'train': str(TRAIN_PATH),
        'val': str(VAL_PATH),
        'test': str(TEST_PATH),
    },
    'output_files': {
        'train_encoded': str(train_output),
        'val_encoded': str(val_output),
        'test_encoded': str(test_output),
    },
    'alert_mapping': alert_mapping,
    'volcano_categories': volcano_categories,
    'encoded_columns': train_encoded.columns.tolist(),
}

with open(ARTIFACT_PATH, 'w', encoding='utf-8') as f:
    json.dump(artifacts, f, indent=2)

timing_output = OUTPUT_DIR / 'process_times_preprocessing.csv'
timing_df = pd.DataFrame(
    [{'process': k, 'duration_seconds': v} for k, v in process_times.items()]
).sort_values('duration_seconds', ascending=False).reset_index(drop=True)
timing_df.to_csv(timing_output, index=False)

process_times['save_outputs'] = perf_counter() - step_start
print(f"Waktu save_outputs: {process_times['save_outputs']:.6f} detik")
print('Train encoded disimpan ke:', train_output)
print('Val encoded disimpan ke  :', val_output)
print('Test encoded disimpan ke :', test_output)
print('Artefak preprocessing disimpan ke:', ARTIFACT_PATH)
print('Timing preprocessing disimpan ke:', timing_output)
timing_df

Waktu save_outputs: 0.146464 detik
Train encoded disimpan ke: preprocessing_outputs_aug_v2\java_ash_train_augmented_encoded.csv
Val encoded disimpan ke  : preprocessing_outputs_aug_v2\java_ash_val_encoded.csv
Test encoded disimpan ke : preprocessing_outputs_aug_v2\java_ash_test_encoded.csv
Artefak preprocessing disimpan ke: preprocessing_outputs_aug_v2\preprocessing_artifacts.json
Timing preprocessing disimpan ke: preprocessing_outputs_aug_v2\process_times_preprocessing.csv


,process,duration_seconds
0,preprocess_encoding,0.170042
1,load_data,0.039766
2,setup_paths,0.000984
